In [3]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [9]:
df2 = pd.read_csv(r"C:\Users\HP\Desktop\csv5csv.csv")

df2['Cust_Order_Count'] = df2.groupby('Customer Key')['Customer Key'].transform('count')
df2['Cust_Avg_Sales'] = df2.groupby('Customer Key')['Sales'].transform('mean')

df2 = df2.drop(columns=['Row ID', 'Customer Key','Order ID'])

categorical_cols = ['Order Priority', 'Category', 'Sub-Category', 'Segment', 'Market', 'Region', 'Day Name', 'Month Name', 'Is Weekend']
df2 = pd.get_dummies(df2, columns=categorical_cols, drop_first=True)

le = LabelEncoder()
df2['Ship Mode'] = le.fit_transform(df2['Ship Mode'])

X = df2.drop(columns=['Ship Mode'])
y = df2['Ship Mode']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

C:\Users\HP\AppData\Roaming\Python\Python39\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [11]:
best_xgb = xgb.XGBClassifier(
    booster="gbtree",
    n_estimators=150,
    learning_rate=0.1,
    max_depth=10,
    reg_alpha=1,
    reg_lambda=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=0
)
best_xgb.fit(X_train_res, y_train_res.to_numpy().ravel())

y_pred_xgb = best_xgb.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred_xgb)
prec = precision_score(y_test, y_pred_xgb, average='weighted' )
rec = recall_score(y_test, y_pred_xgb, average='weighted')
f1 = f1_score(y_test, y_pred_xgb, average='weighted')

print(f"accuracy:  {round(100*acc, 1)}%")
print(f"precision: {round(100*prec, 1)}%")
print(f"recall:    {round(100*rec, 1)}%")
print(f"f1:        {round(100*f1, 1)}%")
print(classification_report(y_test, y_pred_xgb))

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_xgb_df = pd.DataFrame(cm_xgb, index=le.classes_, columns=le.classes_)
cm_xgb_df

accuracy:  85.8%
precision: 84.7%
recall:    85.8%
f1:        84.4%
              precision    recall  f1-score   support

           0       0.76      0.78      0.77      1446
           1       1.00      0.96      0.98       524
           2       0.74      0.47      0.57      1995
           3       0.89      1.00      0.94      5969

    accuracy                           0.86      9934
   macro avg       0.85      0.80      0.81      9934
weighted avg       0.85      0.86      0.84      9934



,First Class,Same Day,Second Class,Standard Class
First Class,1121,0,325,0
Same Day,20,504,0,0
Second Class,342,0,928,725
Standard Class,0,0,3,5966


In [12]:
train_acc = best_xgb.score(X_train_res, y_train_res)
test_acc = best_xgb.score(X_test_scaled, y_test)

print("Train Accuracy:", round(100 * train_acc, 1), "%")
print("Test Accuracy:", round(100 * test_acc, 1), "%")


Train Accuracy: 97.2 %
Test Accuracy: 85.8 %
